# GTFusion on CIC-ToN-IoT

Code for the paper *Graph-Temporal Learning for IoT Intrusion Detection in Centralized and Federated Settings via Adaptive Fusion Gating*.

This notebook runs every experiment reported for **CIC-ToN-IoT**: the centralized comparison, the significance tests, the component ablation, the effect of the centrality features, the per-class analysis, the training cost and time to accuracy, and federated training with five non-IID clients.

**Before running.** Place the dataset file in `data/CIC-ToN-IoT.csv` (see `data/README.md`), install the packages of `requirements.txt`, and run the cells from top to bottom. A GPU is strongly recommended.

N-STGAT is not run on this dataset: its node-sequence encoder processes every node at each step, and its cost grows with the size of the graph (58,295 nodes).

## 1. Dataset choice and imports

The dataset this notebook runs on, and the libraries used.

In [ ]:
DATASET = "CIC-ToN-IoT"
MODE = "FULL"                 # "FAST" or "FULL"
CENT_MODE = "feat"            # centrality version used by GTFusion: "feat" or "guided" (decided after FAST)

import os, json, time, random, copy, math, warnings
from datetime import datetime
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, SAGEConv, GCNConv
from torch_geometric.utils import softmax as pyg_softmax
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, confusion_matrix, precision_recall_fscore_support)
from scipy.stats import chi2, binomtest, ttest_rel

warnings.filterwarnings("ignore")
MODE = os.environ.get("GTF_MODE", MODE)          # only used for testing outside the notebook

## 2. Configuration

Dataset definition, experimental protocol and hyperparameters. `MODE = "FULL"` reproduces the paper (seeds 42, 43 and 44, 20 epochs, 30 federated rounds of one local epoch); `MODE = "FAST"` runs a small stratified sample to check that every part of the pipeline works.

In [ ]:
DATASETS = {
    "CIC-ToN-IoT": dict(           # CICFlowMeter features of ToN-IoT; paper Table 1 caps
        path=os.path.join(os.pardir, "data", "CIC-ToN-IoT.csv"), tag="cicton",
        src="Src IP", dst="Dst IP", label="Attack",
        flow=["Src Port", "Dst Port", "Protocol", "Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts",
              "TotLen Fwd Pkts", "TotLen Bwd Pkts",
              "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
              "Fwd IAT Tot", "Fwd IAT Mean", "Fwd IAT Max", "Fwd IAT Min",
              "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min"],
        log=["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts",
             "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max", "Flow IAT Min",
             "Fwd IAT Tot", "Fwd IAT Mean", "Fwd IAT Max", "Fwd IAT Min",
             "Bwd IAT Tot", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max", "Bwd IAT Min"],
        node=dict(in_bytes="TotLen Bwd Pkts", out_bytes="TotLen Fwd Pkts",
                  in_pkts="Tot Bwd Pkts", out_pkts="Tot Fwd Pkts",
                  proto="Protocol", dport="Dst Port", dur="Flow Duration"),
        caps={"Benign": 500000, "xss": 300000, "password": 300000, "injection": 250000},
        skip={"N-STGAT"}),         # about 30 h per seed on a 58K-node graph; reported on the other two datasets,
}
_DS = DATASETS[DATASET]
DATASET_NAME = DATASET
DATASET_PATH = os.environ.get("GTF_DATA", _DS["path"])
SRC_COL, DST_COL, LABEL_COL = _DS["src"], _DS["dst"], _DS["label"]
FLOW_FEATURES, LOG_FEATURES, NODE_COLS = _DS["flow"], _DS["log"], _DS["node"]
assert set(LOG_FEATURES) <= set(FLOW_FEATURES) and set(NODE_COLS.values()) <= set(FLOW_FEATURES)
MIN_SAMPLES_PER_CLASS = 1000
CLASS_CAPS = _DS["caps"]
DATA_SEED = 42                # fixes which rows are kept by capping and the federated partition

if MODE == "FAST":
    SAMPLE_FRAC, SEEDS, EPOCHS = 0.20, [7], 10
    FED = dict(clients=5, alpha=0.5, rounds=5, local_epochs=1, batch_size=2048, lr=2e-3, mu=0.01)
    OUT_DIR = os.path.join(os.pardir, "results", f"{_DS['tag']}_FAST")
elif MODE == "FULL":
    SAMPLE_FRAC, SEEDS, EPOCHS = None, [42, 43, 44], 20
    FED = dict(clients=5, alpha=0.5, rounds=30, local_epochs=1, batch_size=2048, lr=2e-3, mu=0.01)
    OUT_DIR = os.path.join(os.pardir, "results", f"{_DS['tag']}_FULL")
else:
    raise ValueError("MODE must be FAST or FULL")
assert CENT_MODE in ("feat", "guided")

CFG = dict(epochs=EPOCHS, batch_size=2048, lr=2e-3, weight_decay=1e-4, dropout=0.3, hidden=128,
           test_size=0.2, sched_patience=2, sched_factor=0.5, clip=1.0)
SUBNET_NEIGHBOURS = 4         # each IP is linked to the next 4 IPs of its /24 subnet (paper's graph)
NSTGAT_K = 16                 # N-STGAT: each node's 16 most recent training flows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def banner(s):
    print("\n" + "=" * 110 + "\n" + s + "\n" + "=" * 110, flush=True)

## 3. Data and graph construction

Loading, class filtering and capping, the stratified 80/20 split, the graph built from training flows only (communication edges, /24 subnet edges, padding node for unseen IP addresses), node features, the three centrality measures and the Dirichlet partition of the federated clients.

In [ ]:
_DF = {}


def load_dataset():
    if "df" in _DF:
        return _DF["df"]
    cols = [SRC_COL, DST_COL, LABEL_COL] + FLOW_FEATURES
    log(f"Loading {DATASET_PATH}")
    if DATASET_PATH.lower().endswith(".parquet"):
        df = pd.read_parquet(DATASET_PATH, columns=cols)
    else:
        df = pd.read_csv(DATASET_PATH, usecols=cols)
    log(f"Loaded {len(df):,} rows")
    lab = df[LABEL_COL].astype("string").str.strip()
    n_empty = int((lab.isna() | (lab == "")).sum())
    df = df[~(lab.isna() | (lab == ""))].copy()
    df[LABEL_COL] = lab[df.index].astype(str).values
    if n_empty:
        log(f"Dropped {n_empty:,} rows without a label")
    df[SRC_COL] = df[SRC_COL].fillna("0.0.0.0").astype(str)
    df[DST_COL] = df[DST_COL].fillna("0.0.0.0").astype(str)
    for c in FLOW_FEATURES:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[FLOW_FEATURES] = df[FLOW_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float64)

    counts = df[LABEL_COL].value_counts()
    viable = sorted(counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist())
    dropped = sorted(set(counts.index) - set(viable))
    parts = []
    for c in viable:
        d = df[df[LABEL_COL] == c]
        cap = CLASS_CAPS.get(c)
        if cap is not None and len(d) > cap:
            d = d.sample(n=cap, random_state=DATA_SEED)
        parts.append(d)
    df = pd.concat(parts).sort_index().reset_index(drop=True)
    log(f"Dropped classes with < {MIN_SAMPLES_PER_CLASS} flows: {dropped}")
    if SAMPLE_FRAC is not None:
        keep, _ = train_test_split(np.arange(len(df)), train_size=SAMPLE_FRAC,
                                   stratify=df[LABEL_COL].values, random_state=DATA_SEED)
        df = df.iloc[np.sort(keep)].reset_index(drop=True)
        log(f"FAST mode: stratified {100 * SAMPLE_FRAC:.0f}% sample")
    log(f"Dataset used: {len(df):,} flows, {df[LABEL_COL].nunique()} classes")
    print(df[LABEL_COL].value_counts().to_string(), flush=True)
    _DF["df"] = df
    return df


def flow_matrix(df):
    """10 flow features; log1p on byte, packet and duration counts. Scaling is done later (train statistics)."""
    X = df[FLOW_FEATURES].values.astype(np.float64).copy()
    for j, f in enumerate(FLOW_FEATURES):
        if f in LOG_FEATURES:
            X[:, j] = np.log1p(np.clip(X[:, j], 0, None))
    return X


def subnet_of(ip):
    p = ip.split(".")
    return ".".join(p[:3]) if len(p) == 4 else ip


def graph_parts(train_df):
    """Everything about the graph that comes from TRAINING flows only (raw, before any scaling):
    node list, real edges, subnet edges, raw node features (log1p), raw centrality on the subnet graph,
    node flow positions for N-STGAT."""
    ips = np.unique(np.concatenate([train_df[SRC_COL].values, train_df[DST_COL].values]))
    ip_to_idx = {ip: i for i, ip in enumerate(ips)}
    n = len(ips)
    s = train_df[SRC_COL].map(ip_to_idx).values.astype(np.int64)
    d = train_df[DST_COL].map(ip_to_idx).values.astype(np.int64)
    keep = s != d
    real = np.unique(np.stack([np.minimum(s[keep], d[keep]), np.maximum(s[keep], d[keep])], 1), axis=0) \
        if keep.any() else np.zeros((0, 2), dtype=np.int64)

    groups = {}
    for ip in ips:                                   # ips is sorted, so neighbours are deterministic
        groups.setdefault(subnet_of(ip), []).append(ip_to_idx[ip])
    sub = []
    for members in groups.values():
        for i in range(len(members)):
            for j in range(i + 1, min(i + 1 + SUBNET_NEIGHBOURS, len(members))):
                sub.append((min(members[i], members[j]), max(members[i], members[j])))
    sub = np.array(sub, dtype=np.int64).reshape(-1, 2)
    subnet = np.unique(np.concatenate([real, sub], 0), axis=0) if len(sub) else real

    NC = NODE_COLS
    base = train_df[[NC[k] for k in ("in_bytes", "out_bytes", "in_pkts", "out_pkts", "proto", "dport", "dur")]]
    base = base.reset_index(drop=True)
    base.columns = ["in_bytes", "out_bytes", "in_pkts", "out_pkts", "proto", "dport", "dur"]
    both = pd.concat([base.assign(node=s), base.assign(node=d)], ignore_index=True)
    g = both.groupby("node")
    nf = pd.DataFrame({
        "in_bytes": g["in_bytes"].sum(), "out_bytes": g["out_bytes"].sum(),
        "in_pkts": g["in_pkts"].sum(), "out_pkts": g["out_pkts"].sum(),
        "n_proto": g["proto"].nunique(), "n_dport": g["dport"].nunique(),
        "n_flows": g.size(), "mean_dur": g["dur"].mean(),
    }).reindex(np.arange(n)).fillna(0.0)
    node_raw = np.log1p(np.clip(nf.values.astype(np.float64), 0, None))

    t0 = time.time()
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(subnet.tolist())
    deg = nx.degree_centrality(G)
    btw = nx.betweenness_centrality(G, k=None if n <= 5000 else 1000, seed=0)
    pr = nx.pagerank(G, alpha=0.85)
    cent_time = time.time() - t0
    C = np.array([[deg[i], btw[i], pr[i]] for i in range(n)], dtype=np.float64)
    cent_raw = np.log1p(C * n)                       # size-free scale, then log to tame the heavy tail

    return dict(ip_to_idx=ip_to_idx, n=n, src=s, dst=d, real=real, subnet=subnet,
                node_raw=node_raw, cent_raw=cent_raw, cent_time=cent_time)


def zscore_stats(M):
    mu, sd = M.mean(0), M.std(0)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return mu, sd


def undirected_to_index(E):
    return torch.tensor(np.concatenate([E, E[:, ::-1]], 0).T.copy(), dtype=torch.long)


def nstgat_positions(src, dst, n, K):
    """For every node, positions (in the training-flow array, +1, 0 = padding) of its K most recent flows."""
    pos = np.arange(len(src))
    t = pd.DataFrame({"node": np.r_[src, dst], "pos": np.r_[pos, pos]})
    t = t.sort_values(["node", "pos"])
    t["rk"] = t.groupby("node").cumcount(ascending=False)
    t = t[t["rk"] < K]
    out = np.zeros((n + 1, K), dtype=np.int64)
    out[t["node"].values, K - 1 - t["rk"].values] = t["pos"].values + 1
    return out


def louvain(E, n):
    G = nx.Graph(); G.add_nodes_from(range(n)); G.add_edges_from(E.tolist())
    try:
        comms = nx.community.louvain_communities(G, seed=0)
    except AttributeError:
        comms = list(nx.algorithms.community.greedy_modularity_communities(G))
    comm = np.zeros(n + 1, dtype=np.int64)
    for ci, c in enumerate(comms):
        comm[list(c)] = ci
    n_comm = len(comms) + 1
    comm[n] = n_comm - 1                             # padding node: its own community
    ei = undirected_to_index(E).numpy()
    cs, cd = comm[ei[0]], comm[ei[1]]
    m = cs != cd
    cp = np.unique(np.stack([cs[m], cd[m]], 1), axis=0) if m.any() else np.zeros((0, 2), dtype=np.int64)
    return torch.tensor(comm), torch.tensor(cp.T.copy(), dtype=torch.long).reshape(2, -1), n_comm, len(comms)


def make_graphs(P, node_mu, node_sd, cent_mu, cent_sd, flows_train_pad):
    """Three graphs over the same nodes. The last node is a padding node (unseen IPs), all zeros, no edges.
       plain       : real edges, 8 node features                       (baselines)
       subnet      : real + subnet edges, 8 node features               (GTFusion ablation without centrality)
       subnet_cent : real + subnet edges, 8 + 3 centrality features,
                     edge_attr = centrality of the message source, score = PageRank (GTFusion)"""
    n = P["n"]
    xb = (P["node_raw"] - node_mu) / node_sd
    xc = (P["cent_raw"] - cent_mu) / cent_sd
    pad8, pad3 = np.zeros((1, 8)), np.zeros((1, 3))
    x8 = torch.tensor(np.concatenate([xb, pad8]), dtype=torch.float)
    c3 = torch.tensor(np.concatenate([xc, pad3]), dtype=torch.float)
    ei_plain, ei_sub = undirected_to_index(P["real"]), undirected_to_index(P["subnet"])

    plain = Data(x=x8, edge_index=ei_plain)
    pos = nstgat_positions(P["src"], P["dst"], n, NSTGAT_K)
    plain.node_seq = flows_train_pad[torch.tensor(pos)]                     # [n+1, K, flow_dim]
    plain.comm, plain.comm_ei, plain.n_comm, n_louvain = louvain(P["real"], n)

    subnet = Data(x=x8, edge_index=ei_sub)
    sc = Data(x=torch.cat([x8, c3], 1), edge_index=ei_sub)
    sc.edge_attr = c3[ei_sub[0]].clone()
    sc.score = c3[:, 2].clone()
    info = dict(n_nodes=n, edges_plain=int(len(P["real"])), edges_subnet=int(len(P["subnet"])),
                louvain_communities=n_louvain, centrality_time=P["cent_time"])
    return dict(plain=plain, subnet=subnet, subnet_cent=sc), info


def map_ips(df_part, ip_to_idx, pad):
    s = df_part[SRC_COL].map(ip_to_idx)
    d = df_part[DST_COL].map(ip_to_idx)
    unseen = float((s.isna() | d.isna()).mean())
    return (torch.tensor(s.fillna(pad).values.astype(np.int64)),
            torch.tensor(d.fillna(pad).values.astype(np.int64)), unseen)


def balanced_weights(counts):
    counts = np.asarray(counts, dtype=np.float64)
    C = len(counts)
    return np.where(counts > 0, counts.sum() / (C * np.maximum(counts, 1)), 0.0)


_DATA = {}


def prepare_centralized(seed):
    if seed in _DATA:
        return _DATA[seed]
    _DATA.clear()
    df = load_dataset()
    le = LabelEncoder().fit(df[LABEL_COL])
    y = le.transform(df[LABEL_COL]).astype(np.int64)      # int64: Windows numpy defaults to int32
    tr, te = train_test_split(np.arange(len(df)), test_size=CFG["test_size"], stratify=y, random_state=seed)
    tr, te = np.sort(tr), np.sort(te)
    assert len(np.intersect1d(tr, te)) == 0
    train_df, test_df = df.iloc[tr], df.iloc[te]

    X = flow_matrix(df)
    f_mu, f_sd = zscore_stats(X[tr])
    Xs = ((X - f_mu) / f_sd).astype(np.float32)
    Xtr, Xte = torch.tensor(Xs[tr]), torch.tensor(Xs[te])
    Xtr_pad = torch.cat([torch.zeros(1, Xtr.shape[1]), Xtr])

    P = graph_parts(train_df)
    node_mu, node_sd = zscore_stats(P["node_raw"])
    cent_mu, cent_sd = zscore_stats(P["cent_raw"])
    graphs, ginfo = make_graphs(P, node_mu, node_sd, cent_mu, cent_sd, Xtr_pad)
    pad = P["n"]
    s_tr, d_tr, u_tr = map_ips(train_df, P["ip_to_idx"], pad)
    s_te, d_te, u_te = map_ips(test_df, P["ip_to_idx"], pad)
    assert u_tr == 0.0, "a training flow has an IP outside the training graph"
    assert (s_tr.numpy() == P["src"]).all() and (d_tr.numpy() == P["dst"]).all()

    D = dict(train=dict(src=s_tr, dst=d_tr, x=Xtr, y=torch.tensor(y[tr], dtype=torch.long), n=len(tr)),
             test=dict(src=s_te, dst=d_te, x=Xte, y=torch.tensor(y[te], dtype=torch.long), n=len(te), unseen=u_te),
             graphs=graphs, classes=le.classes_.tolist(), n_classes=len(le.classes_), flow_dim=Xs.shape[1],
             class_weights=torch.tensor(balanced_weights(np.bincount(y[tr], minlength=len(le.classes_))),
                                        dtype=torch.float), **ginfo)
    log(f"Split seed {seed}: train {len(tr):,} / test {len(te):,} | {ginfo['n_nodes']:,} nodes | "
        f"edges plain {ginfo['edges_plain']:,}, with subnet {ginfo['edges_subnet']:,} | "
        f"centrality {ginfo['centrality_time']:.2f}s | test flows with an unseen IP {100 * u_te:.2f}%")
    _DATA[seed] = D
    return D

## 4. Models

All compared models: LSTM, GAT, GraphSAGE, FN-GNN, Sequential GAT-LSTM, N-STGAT, FedGATSage, the same-depth baselines GAT-3L and GraphSAGE-3L, the two GTFusion variants and the ablation variants.

In [ ]:
# Classifier head and gate use BatchNorm (the validated configuration). In federated training the BatchNorm
# layers stay LOCAL on each client (FedBN, Li et al. 2021), so no running statistics are mixed across clients.
def mlp_head(in_dim, n_classes, dropout):
    return nn.Sequential(
        nn.Linear(in_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(dropout),
        nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(dropout),
        nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
        nn.Linear(128, n_classes))


GAT_HEADS = {2: [8, 4], 3: [8, 8, 4], 4: [8, 8, 4, 2]}


class CentSAGEConv(nn.Module):
    """GraphSAGE layer with a centrality-weighted neighbour mean: weight_j = softmax(tau * pagerank_j).
    tau is learned and starts at 1 (tau = 0 is the plain mean)."""
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin_root, self.lin_nb = nn.Linear(d_in, d_out), nn.Linear(d_in, d_out, bias=False)
        self.tau = nn.Parameter(torch.tensor(1.0))

    def forward(self, x, edge_index, score):
        s, d = edge_index
        w = pyg_softmax(self.tau * score[s], d, num_nodes=x.size(0))
        agg = torch.zeros(x.size(0), x.size(1), device=x.device, dtype=x.dtype).index_add_(0, d, x[s] * w.unsqueeze(1))
        return self.lin_root(x) + self.lin_nb(agg)


class GNNEncoder(nn.Module):
    """kind gat/sage. guided=True: GAT attention uses edge centrality features, SAGE uses CentSAGEConv."""
    def __init__(self, in_dim, kind, n_layers, hidden=128, dropout=0.3, guided=False):
        super().__init__()
        self.kind, self.guided, self.dropout = kind, guided, dropout
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for i in range(n_layers):
            d_in = in_dim if i == 0 else hidden
            if kind == "gat":
                h = GAT_HEADS[n_layers][i]
                self.convs.append(GATConv(d_in, hidden // h, heads=h, dropout=dropout,
                                          edge_dim=3 if guided else None))
            elif kind == "sage":
                self.convs.append(CentSAGEConv(d_in, hidden) if guided else SAGEConv(d_in, hidden))
            else:
                raise ValueError(kind)
            self.norms.append(nn.LayerNorm(hidden))

    def forward(self, g):
        x, last = g.x, len(self.convs) - 1
        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            if self.guided and self.kind == "gat":
                x = conv(x, g.edge_index, edge_attr=g.edge_attr)
            elif self.guided:
                x = conv(x, g.edge_index, g.score)
            else:
                x = conv(x, g.edge_index)
            x = norm(x)
            if i < last:
                x = F.dropout(F.relu(x), p=self.dropout, training=self.training)
        return x


class TemporalEncoder(nn.Module):
    """LSTM encoder (default 2-layer BiLSTM). Input [batch, steps, dim]. Output out_dim = hidden * (2 if bi else 1)."""
    def __init__(self, in_dim, hidden=128, dropout=0.3, layers=2, bidirectional=True):
        super().__init__()
        self.bi = bidirectional
        self.lstm = nn.LSTM(in_dim, hidden, num_layers=layers, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0, bidirectional=bidirectional)
        self.out_dim = hidden * (2 if bidirectional else 1)
        self.norm = nn.LayerNorm(self.out_dim)

    def forward(self, seq):
        _, (h, _) = self.lstm(seq)
        return self.norm(torch.cat([h[-2], h[-1]], dim=1) if self.bi else h[-1])


class LSTMModel(nn.Module):
    """Baseline: flow features only."""
    graph_key = None
    def __init__(self, node_dim, flow_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.temporal = TemporalEncoder(flow_dim, hidden, dropout)
        self.head = mlp_head(2 * hidden, n_classes, dropout)

    def node_emb(self, g):
        return None

    def forward(self, g, src, dst, x, emb=None):
        return self.head(self.temporal(x.unsqueeze(1)))


class GNNModel(nn.Module):
    """Baseline: GAT or GraphSAGE, graph only. Flow = [source embedding, destination embedding]."""
    graph_key = "plain"
    def __init__(self, node_dim, flow_dim, n_classes, kind="gat", n_layers=4, hidden=128, dropout=0.3):
        super().__init__()
        self.enc = GNNEncoder(node_dim, kind, n_layers, hidden, dropout)
        self.head = mlp_head(2 * hidden, n_classes, dropout)

    def node_emb(self, g):
        return self.enc(g)

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        return self.head(torch.cat([e[src], e[dst]], 1))


class FNGNNModel(nn.Module):
    """Baseline: FN-GNN (Tran and Park), graph only. Local feature path + GCN/SAGE neighbourhood path."""
    graph_key = "plain"
    def __init__(self, node_dim, flow_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        self.local = nn.Sequential(nn.Linear(node_dim, hidden), nn.ReLU(), nn.LayerNorm(hidden), nn.Dropout(dropout),
                                   nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.LayerNorm(hidden // 2))
        self.gcn, self.sage = GCNConv(node_dim, hidden), SAGEConv(hidden, hidden // 2)
        self.n1, self.n2 = nn.LayerNorm(hidden), nn.LayerNorm(hidden // 2)
        self.head = mlp_head(2 * hidden, n_classes, dropout)

    def node_emb(self, g):
        h = F.dropout(self.n1(F.relu(self.gcn(g.x, g.edge_index))), p=self.dropout, training=self.training)
        return torch.cat([self.local(g.x), self.n2(F.relu(self.sage(h, g.edge_index)))], 1)

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        return self.head(torch.cat([e[src], e[dst]], 1))


class SeqGATLSTM(nn.Module):
    """Baseline: sequential spatial then temporal. GAT endpoint embeddings are passed through the LSTM."""
    graph_key = "plain"
    def __init__(self, node_dim, flow_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.enc = GNNEncoder(node_dim, "gat", 3, hidden, dropout)
        self.temporal = TemporalEncoder(2 * hidden, hidden, dropout)
        self.head = mlp_head(2 * hidden, n_classes, dropout)

    def node_emb(self, g):
        return self.enc(g)

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        return self.head(self.temporal(torch.cat([e[src], e[dst]], 1).unsqueeze(1)))


class NSTGAT(nn.Module):
    """Baseline: N-STGAT style (Wang et al.), temporal then spatial. An LSTM encodes each node's recent flows,
    the result joins the node features and goes through GAT layers. Flow = [src emb, dst emb, flow features]."""
    graph_key = "plain"
    def __init__(self, node_dim, flow_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.temporal = TemporalEncoder(flow_dim, hidden, dropout)
        self.enc = GNNEncoder(node_dim + 2 * hidden, "gat", 3, hidden, dropout)
        self.head = mlp_head(2 * hidden + flow_dim, n_classes, dropout)

    def node_emb(self, g):
        h = torch.cat([g.x, self.temporal(g.node_seq)], 1)
        return self.enc(Data(x=h, edge_index=g.edge_index))

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        return self.head(torch.cat([e[src], e[dst], x], 1))


class FedGATSageModel(nn.Module):
    """Baseline: FedGATSage architecture (centralized form). GAT on the device graph, Louvain community
    abstraction, GraphSAGE on the community graph. Flow = [src emb, dst emb, flow features]."""
    graph_key = "plain"
    def __init__(self, node_dim, flow_dim, n_classes, hidden=128, dropout=0.3):
        super().__init__()
        self.hidden = hidden
        self.gat = GNNEncoder(node_dim, "gat", 2, hidden, dropout)
        self.sage = GNNEncoder(hidden, "sage", 2, hidden, dropout)
        self.head = mlp_head(4 * hidden + flow_dim, n_classes, dropout)

    def node_emb(self, g):
        h = self.gat(g)
        tot = torch.zeros(g.n_comm, self.hidden, device=h.device).index_add_(0, g.comm, h)
        cnt = torch.zeros(g.n_comm, device=h.device).index_add_(0, g.comm, torch.ones(len(g.comm), device=h.device))
        c = self.sage(Data(x=tot / cnt.clamp(min=1).unsqueeze(1), edge_index=g.comm_ei))
        return torch.cat([h, c[g.comm]], 1)

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        return self.head(torch.cat([e[src], e[dst], x], 1))


class GTFusion(nn.Module):
    """Parallel GNN branch and BiLSTM branch.
    graph : 'plain' | 'subnet' | 'subnet_cent'   cent: None | 'feat' | 'guided'   fusion: 'gate' | 'concat'
    gate  : MLP on [g, t] -> softmax (w_graph, w_temporal); fused = [w_graph * g, w_temporal * t]."""
    def __init__(self, node_dim, flow_dim, n_classes, kind="gat", n_layers=3, graph="subnet_cent",
                 cent="feat", fusion="gate", lstm_layers=2, lstm_bi=True, hidden=128, dropout=0.3):
        super().__init__()
        self.graph_key, self.fusion = graph, fusion
        self.enc = GNNEncoder(node_dim, kind, n_layers, hidden, dropout, guided=(cent == "guided"))
        self.temporal = TemporalEncoder(flow_dim, hidden, dropout, layers=lstm_layers, bidirectional=lstm_bi)
        d = 2 * hidden + self.temporal.out_dim
        if fusion == "gate":
            self.gate = nn.Sequential(nn.Linear(d, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(dropout),
                                      nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(dropout),
                                      nn.Linear(128, 2))
        self.head = mlp_head(d, n_classes, dropout)
        self.last_w = None

    def node_emb(self, g):
        return self.enc(g)

    def forward(self, g, src, dst, x, emb=None):
        e = self.node_emb(g) if emb is None else emb
        gv = torch.cat([e[src], e[dst]], 1)
        tv = self.temporal(x.unsqueeze(1))
        cat = torch.cat([gv, tv], 1)
        if self.fusion == "gate":
            w = torch.softmax(self.gate(cat), 1)
            self.last_w = w.detach()
            fused = torch.cat([w[:, :1] * gv, w[:, 1:] * tv], 1)
        else:
            self.last_w = None
            fused = cat
        return self.head(fused)


ALT_CENT = "guided" if CENT_MODE == "feat" else "feat"
MODELS = OrderedDict([
    # ---- main table: basic baselines
    ("LSTM",          (LSTMModel, {})),
    ("GAT",           (GNNModel, dict(kind="gat", n_layers=4))),
    ("GraphSAGE",     (GNNModel, dict(kind="sage", n_layers=4))),
    ("FN-GNN",        (FNGNNModel, {})),
    ("Seq-GAT-LSTM",  (SeqGATLSTM, {})),
    ("N-STGAT",       (NSTGAT, {})),
    ("FedGATSage",    (FedGATSageModel, {})),
    # ---- main table: GTFusion, full pipeline
    ("GTFusion-GAT",  (GTFusion, dict(kind="gat", graph="subnet_cent", cent=CENT_MODE, fusion="gate"))),
    ("GTFusion-SAGE", (GTFusion, dict(kind="sage", graph="subnet_cent", cent=CENT_MODE, fusion="gate"))),
    # ---- leave-one-out ablation (each removes one part of our pipeline)
    ("GTFusion-GAT -centrality",         (GTFusion, dict(kind="gat", graph="subnet", cent=None, fusion="gate"))),
    ("GTFusion-GAT -centrality -subnet", (GTFusion, dict(kind="gat", graph="plain", cent=None, fusion="gate"))),
    ("GTFusion-GAT -gate",               (GTFusion, dict(kind="gat", graph="subnet_cent", cent=CENT_MODE, fusion="concat"))),
    (f"GTFusion-GAT cent={ALT_CENT}",    (GTFusion, dict(kind="gat", graph="subnet_cent", cent=ALT_CENT, fusion="gate"))),
    ("GTFusion-SAGE -centrality",         (GTFusion, dict(kind="sage", graph="subnet", cent=None, fusion="gate"))),
    ("GTFusion-SAGE -centrality -subnet", (GTFusion, dict(kind="sage", graph="plain", cent=None, fusion="gate"))),
    ("GTFusion-SAGE -gate",               (GTFusion, dict(kind="sage", graph="subnet_cent", cent=CENT_MODE, fusion="concat"))),
    (f"GTFusion-SAGE cent={ALT_CENT}",    (GTFusion, dict(kind="sage", graph="subnet_cent", cent=ALT_CENT, fusion="gate"))),
    # ---- matched depth: standalone GNNs with the SAME depth and heads as the GTFusion graph branch (3 layers, 8/8/4)
    ("GAT-3L",       (GNNModel, dict(kind="gat", n_layers=3))),
    ("GraphSAGE-3L", (GNNModel, dict(kind="sage", n_layers=3))),
])
if MODE == "FULL":                       # the centrality version is decided in FAST; FULL runs only the chosen one
    for _k in ("GAT", "SAGE"):
        MODELS.pop(f"GTFusion-{_k} cent={ALT_CENT}")
BASELINES = ["LSTM", "GAT", "GraphSAGE", "FN-GNN", "Seq-GAT-LSTM", "N-STGAT", "FedGATSage"]
OURS = ["GTFusion-GAT", "GTFusion-SAGE"]
MAIN = BASELINES + OURS
ABLATION = {k: [m for m in [f"GTFusion-{k}", f"GTFusion-{k} -gate", f"GTFusion-{k} -centrality",
                             f"GTFusion-{k} -centrality -subnet", f"GTFusion-{k} cent={ALT_CENT}"] if m in MODELS]
            for k in ("GAT", "SAGE")}
FED_MODELS = ["LSTM", "GAT", "GraphSAGE", "FN-GNN", "Seq-GAT-LSTM", "N-STGAT", "FedGATSage",
              "GTFusion-GAT", "GTFusion-SAGE"]


def graph_key_of(name):
    cls, kw = MODELS[name]
    return kw.get("graph", getattr(cls, "graph_key", None)) if cls is GTFusion else cls.graph_key


def build_model(name, node_dims, flow_dim, n_classes):
    cls, kw = MODELS[name]
    gk = graph_key_of(name)
    node_dim = node_dims[gk] if gk is not None else 0
    return cls(node_dim, flow_dim, n_classes, hidden=CFG["hidden"], dropout=CFG["dropout"], **kw)

## 5. Centralized training and evaluation

Centralized training with a class-weighted cross-entropy loss and AdamW, a fixed number of epochs, and evaluation of the final model.

In [ ]:
def model_graph(model_name, graphs):
    gk = graph_key_of(model_name)
    return graphs[gk].to(DEVICE) if gk is not None else None


@torch.no_grad()
def predict(model, g, split, batch_size=8192):
    model.eval()
    emb = model.node_emb(g) if g is not None else None
    preds, ws = [], []
    for i in range(0, split["n"], batch_size):
        sl = slice(i, min(i + batch_size, split["n"]))
        out = model(g, split["src"][sl].to(DEVICE), split["dst"][sl].to(DEVICE), split["x"][sl].to(DEVICE), emb=emb)
        preds.append(out.argmax(1).cpu())
        if getattr(model, "last_w", None) is not None:
            ws.append(model.last_w.cpu())
    return torch.cat(preds).numpy(), (torch.cat(ws).numpy() if ws else None)


def compute_metrics(y_true, y_pred, classes):
    labels = np.arange(len(classes))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    tp = np.diag(cm).astype(float)
    fn, fp = cm.sum(1) - tp, cm.sum(0) - tp
    tn = cm.sum() - tp - fn - fp
    fnr = np.divide(fn, fn + tp, out=np.zeros_like(tp), where=(fn + tp) > 0)
    fpr = np.divide(fp, fp + tn, out=np.zeros_like(tp), where=(fp + tn) > 0)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    m = dict(accuracy=float(accuracy_score(y_true, y_pred)),
             balanced_accuracy=float(balanced_accuracy_score(y_true, y_pred)),
             f1_weighted=float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
             f1_macro=float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
             precision_weighted=float(precision_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
             recall_weighted=float(recall_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
             fpr_macro=float(fpr.mean()), fnr_macro=float(fnr.mean()),
             per_class={c: dict(precision=float(p[i]), recall=float(r[i]), f1=float(f[i]), fpr=float(fpr[i]),
                                fnr=float(fnr[i]), support=int(s[i])) for i, c in enumerate(classes)},
             confusion_matrix=cm.tolist())
    present = s > 0
    assert abs(m["recall_weighted"] - m["accuracy"]) < 1e-9
    if present.all():
        assert abs(r.mean() - m["balanced_accuracy"]) < 1e-9
    return m


def _safe(name):
    return "".join(ch if ch.isalnum() or ch in "-_=" else "_" for ch in name)


def _save(base, res, arrays):
    np.savez_compressed(base + ".npz", **arrays)
    with open(base + ".json", "w") as f:
        json.dump(res, f)


def gate_summary(w, y_true, classes):
    if w is None:
        return None
    assert np.allclose(w.sum(1), 1.0, atol=1e-4)
    out = {c: dict(w_graph=float(w[y_true == i, 0].mean()), w_graph_std=float(w[y_true == i, 0].std()),
                   n=int((y_true == i).sum())) for i, c in enumerate(classes) if (y_true == i).any()}
    out["_overall"] = dict(w_graph=float(w[:, 0].mean()), w_graph_std=float(w[:, 0].std()), n=int(len(w)))
    return out


def run_central(name, seed, tag="central"):
    base = os.path.join(RUN_DIR, f"{tag}__{_safe(name)}__seed{seed}")
    if os.path.exists(base + ".json"):
        with open(base + ".json") as f:
            return json.load(f)
    D = prepare_centralized(seed)
    node_dims = {k: g.x.shape[1] for k, g in D["graphs"].items()}
    set_seed(seed)
    model = build_model(name, node_dims, D["flow_dim"], D["n_classes"]).to(DEVICE)
    g = model_graph(name, D["graphs"])
    n_params = int(sum(p.numel() for p in model.parameters()))
    log(f">>> {name} | seed {seed} | graph {graph_key_of(name)} | {n_params:,} parameters")

    tr, te = D["train"], D["test"]
    yt = te["y"].numpy()
    crit = nn.CrossEntropyLoss(weight=D["class_weights"].to(DEVICE))
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=CFG["sched_factor"],
                                                       patience=CFG["sched_patience"])
    gen = torch.Generator().manual_seed(seed)
    history, train_time = [], 0.0
    for ep in range(1, CFG["epochs"] + 1):
        model.train()
        t0 = time.time()
        perm = torch.randperm(tr["n"], generator=gen)
        loss_sum, seen = 0.0, 0
        for i in range(0, tr["n"], CFG["batch_size"]):
            idx = perm[i:i + CFG["batch_size"]]
            if len(idx) < 2:
                continue
            out = model(g, tr["src"][idx].to(DEVICE), tr["dst"][idx].to(DEVICE), tr["x"][idx].to(DEVICE))
            loss = crit(out, tr["y"][idx].to(DEVICE))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"])
            opt.step()
            loss_sum += loss.item() * len(idx); seen += len(idx)
        ep_time = time.time() - t0
        train_time += ep_time
        tl = loss_sum / max(seen, 1)
        assert np.isfinite(tl), "training loss is NaN or infinite: stop and report"
        sched.step(tl)                                        # schedule on TRAINING loss only
        yp, _ = predict(model, g, te)                         # logged for curves only, never used to select
        history.append(dict(epoch=ep, train_loss=tl, test_accuracy=float(accuracy_score(yt, yp)),
                            test_balanced_accuracy=float(balanced_accuracy_score(yt, yp)),
                            lr=opt.param_groups[0]["lr"], epoch_time=ep_time, cumulative_time=train_time))
        log(f"    epoch {ep:2d}/{CFG['epochs']}  loss {tl:.4f}  test_acc {history[-1]['test_accuracy']:.4f}  "
            f"test_bal {history[-1]['test_balanced_accuracy']:.4f}  ({ep_time:.0f}s)")

    t0 = time.time()
    yp, w = predict(model, g, te)                             # FINAL model, no selection
    infer = time.time() - t0
    m = compute_metrics(yt, yp, D["classes"])
    res = dict(model=name, seed=seed, tag=tag, mode=MODE, cent_mode=CENT_MODE, dataset=DATASET_NAME,
               config=CFG, n_params=n_params, graph=graph_key_of(name), n_train=tr["n"], n_test=te["n"],
               n_nodes=D["n_nodes"], edges_plain=D["edges_plain"], edges_subnet=D["edges_subnet"],
               centrality_time=D["centrality_time"], test_unseen_ip_rate=te["unseen"],
               train_time=train_time, inference_time=infer, history=history, metrics=m,
               gate=gate_summary(w, yt, D["classes"]), classes=D["classes"],
               finished=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    _save(base, res, dict(y_true=yt, y_pred=yp, gate_w=w if w is not None else np.zeros((0, 2))))
    gtxt = f" | gate w_graph {w[:, 0].mean():.3f}" if w is not None else ""
    log(f"<<< {name} | acc {m['accuracy']:.4f} | bal {m['balanced_accuracy']:.4f} | F1w {m['f1_weighted']:.4f} | "
        f"F1m {m['f1_macro']:.4f} | train {train_time:.0f}s{gtxt}")
    return res

## 6. Federated training

Federated training: FedAvg on the shared parameters, FedProx on each client, batch normalization layers and the gate kept local on each client (FedBN and personalized gate).

In [ ]:
#   Dirichlet(alpha) label skew over 5 clients, fixed partition (DATA_SEED). Each client: own 80/20 split and own
#   graph from its own training flows. Shared statistics, built from client sums only (no raw data leaves a client):
#   flow scaler, node feature scaler, centrality scaler, class weights.
#   FedAvg weighted by client size + FedProx (mu). Cosine learning-rate decay over rounds. Final round reported.
#   GTFusion keeps its gate LOCAL on each client (personalised gate); everything else is averaged.
_FEDDATA = {}


def _pooled(stats):
    """Combine per-client (count, sum, sum of squares) into a global mean and std."""
    n = sum(s[0] for s in stats)
    mu = sum(s[1] for s in stats) / n
    var = sum(s[2] for s in stats) / n - mu ** 2
    sd = np.sqrt(np.clip(var, 0, None))
    return mu, np.where(sd < 1e-8, 1.0, sd)


def _moments(M):
    return (len(M), M.sum(0), (M ** 2).sum(0))


def prepare_federated():
    if "clients" in _FEDDATA:
        return _FEDDATA["clients"], _FEDDATA["classes"], _FEDDATA["info"]
    df = load_dataset()
    le = LabelEncoder().fit(df[LABEL_COL])
    y = le.transform(df[LABEL_COL]).astype(np.int64)      # int64: Windows numpy defaults to int32
    C, K = len(le.classes_), FED["clients"]
    rng = np.random.default_rng(DATA_SEED)
    for attempt in range(200):
        owner = np.empty(len(df), dtype=np.int64)
        for c in range(C):
            idx = np.where(y == c)[0]
            rng.shuffle(idx)
            p = rng.dirichlet([FED["alpha"]] * K)
            cuts = (np.cumsum(p)[:-1] * len(idx)).astype(int)
            for k, part in enumerate(np.split(idx, cuts)):
                owner[part] = k
        if np.bincount(owner, minlength=K).min() >= 0.02 * len(df):
            break
    X = flow_matrix(df)
    raw = []
    for k in range(K):
        ix = np.where(owner == k)[0]
        ix = ix[np.random.default_rng(DATA_SEED + 1 + k).permutation(len(ix))]
        n_te = int(round(0.2 * len(ix)))
        te, tr = np.sort(ix[:n_te]), np.sort(ix[n_te:])
        P = graph_parts(df.iloc[tr])
        raw.append(dict(tr=tr, te=te, P=P))

    # shared statistics from client moments (equal to fitting on the union of all client training data)
    f_mu, f_sd = _pooled([_moments(X[r["tr"]]) for r in raw])
    n_mu, n_sd = _pooled([_moments(r["P"]["node_raw"]) for r in raw])
    c_mu, c_sd = _pooled([_moments(r["P"]["cent_raw"]) for r in raw])
    counts = sum(np.bincount(y[r["tr"]], minlength=C) for r in raw)
    cw = torch.tensor(balanced_weights(counts), dtype=torch.float)
    union = np.concatenate([r["tr"] for r in raw])
    assert np.allclose(f_mu, X[union].mean(0)) and np.allclose(f_sd, np.where(X[union].std(0) < 1e-8, 1, X[union].std(0)))

    Xs = ((X - f_mu) / f_sd).astype(np.float32)
    clients = []
    for k, r in enumerate(raw):
        Xtr, Xte = torch.tensor(Xs[r["tr"]]), torch.tensor(Xs[r["te"]])
        graphs, ginfo = make_graphs(r["P"], n_mu, n_sd, c_mu, c_sd, torch.cat([torch.zeros(1, Xtr.shape[1]), Xtr]))
        s_tr, d_tr, _ = map_ips(df.iloc[r["tr"]], r["P"]["ip_to_idx"], r["P"]["n"])
        s_te, d_te, u_te = map_ips(df.iloc[r["te"]], r["P"]["ip_to_idx"], r["P"]["n"])
        cc = np.bincount(y[r["tr"]], minlength=C)
        clients.append(dict(train=dict(src=s_tr, dst=d_tr, x=Xtr, y=torch.tensor(y[r["tr"]], dtype=torch.long), n=len(r["tr"])),
                            test=dict(src=s_te, dst=d_te, x=Xte, y=torch.tensor(y[r["te"]], dtype=torch.long), n=len(r["te"]), unseen=u_te),
                            graphs=graphs, class_counts=cc.tolist(), **ginfo))
        log(f"client {k}: {len(r['tr']):,} train / {len(r['te']):,} test | {ginfo['n_nodes']:,} nodes | "
            f"edges plain {ginfo['edges_plain']:,}, subnet {ginfo['edges_subnet']:,} | unseen IP {100 * u_te:.2f}% | "
            f"classes {int((cc > 0).sum())}/{C} | per class {cc.tolist()}")
    info = dict(class_weights=cw, flow_dim=X.shape[1], n_classes=C)
    _FEDDATA.update(clients=clients, classes=le.classes_.tolist(), info=info)
    return clients, le.classes_.tolist(), info


def local_key_set(model, model_name):
    """Keys kept on each client: every BatchNorm layer (FedBN, all models) and the GTFusion gate (personalised)."""
    bn = [n for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d)]
    keys = set()
    for k in model.state_dict():
        if any(k.startswith(b + ".") for b in bn) or (model_name.startswith("GTFusion") and k.startswith("gate.")):
            keys.add(k)
    return keys


def run_fed(name, seed, tag="federated"):
    base = os.path.join(RUN_DIR, f"{tag}__{_safe(name)}__seed{seed}")
    if os.path.exists(base + ".json"):
        with open(base + ".json") as f:
            return json.load(f)
    clients, classes, info = prepare_federated()
    node_dims = {k: g.x.shape[1] for k, g in clients[0]["graphs"].items()}
    set_seed(seed)
    model = build_model(name, node_dims, info["flow_dim"], info["n_classes"]).to(DEVICE)
    n_params = int(sum(p.numel() for p in model.parameters()))
    log(f">>> FED {name} | seed {seed} | {len(clients)} clients | {n_params:,} parameters")
    crit = nn.CrossEntropyLoss(weight=info["class_weights"].to(DEVICE))
    gen = torch.Generator().manual_seed(seed)
    sizes = np.array([c["train"]["n"] for c in clients], dtype=np.float64)
    wts = sizes / sizes.sum()
    global_state = copy.deepcopy(model.state_dict())
    LOCAL = local_key_set(model, name)
    local_keys = [k for k in global_state if k in LOCAL]
    local_states = [{k: global_state[k].clone() for k in local_keys} for _ in clients]
    graphs = [model_graph(name, c["graphs"]) for c in clients]
    history, t_train = [], 0.0
    R = FED["rounds"]
    for rnd in range(1, R + 1):
        lr = FED["lr"] * (0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (rnd - 1) / R)))
        t0 = time.time()
        new_states = []
        for k, c in enumerate(clients):
            st = copy.deepcopy(global_state); st.update(local_states[k])
            model.load_state_dict(st)
            anchor = {n_: p.detach().clone() for n_, p in model.named_parameters() if n_ not in LOCAL}
            opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["weight_decay"])
            model.train()
            tr = c["train"]
            for _ in range(FED["local_epochs"]):
                perm = torch.randperm(tr["n"], generator=gen)
                for i in range(0, tr["n"], FED["batch_size"]):
                    idx = perm[i:i + FED["batch_size"]]
                    if len(idx) < 2:
                        continue
                    out = model(graphs[k], tr["src"][idx].to(DEVICE), tr["dst"][idx].to(DEVICE), tr["x"][idx].to(DEVICE))
                    loss = crit(out, tr["y"][idx].to(DEVICE))
                    if FED["mu"] > 0:
                        prox = sum(((p - anchor[n_]) ** 2).sum() for n_, p in model.named_parameters() if n_ in anchor)
                        loss = loss + 0.5 * FED["mu"] * prox
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["clip"])
                    opt.step()
                    assert np.isfinite(loss.item()), "loss is NaN or infinite: stop and report"
            sd_k = copy.deepcopy(model.state_dict())
            local_states[k] = {kk: sd_k[kk].clone() for kk in local_keys}
            new_states.append(sd_k)
        for key in global_state:
            if key in local_keys:
                continue
            if torch.is_floating_point(global_state[key]):
                global_state[key] = sum(s[key] * float(w) for s, w in zip(new_states, wts))
        t_train += time.time() - t0
        yt, yp, _, _ = _fed_predict(model, name, clients, graphs, global_state, local_states)
        history.append(dict(round=rnd, lr=lr, test_accuracy=float(accuracy_score(yt, yp)),
                            test_balanced_accuracy=float(balanced_accuracy_score(yt, yp)), cumulative_time=t_train))
        log(f"    round {rnd:2d}/{R}  lr {lr:.2e}  test_acc {history[-1]['test_accuracy']:.4f}  "
            f"test_bal {history[-1]['test_balanced_accuracy']:.4f}  ({time.time() - t0:.0f}s)")

    yt, yp, w, cid = _fed_predict(model, name, clients, graphs, global_state, local_states)   # FINAL round
    m = compute_metrics(yt, yp, classes)
    per_client = {}
    for k in range(len(clients)):
        mk = cid == k
        per_client[f"client_{k}"] = dict(
            n_test=int(mk.sum()), accuracy=float(accuracy_score(yt[mk], yp[mk])),
            class_counts_train=clients[k]["class_counts"],
            gate=gate_summary(w[mk], yt[mk], classes) if w is not None else None)
    gate_spread = None
    gate_keys = [k_ for k_ in local_keys if k_.startswith("gate.") and torch.is_floating_point(global_state[k_])]
    if gate_keys:
        flat = [torch.cat([ls[k_].flatten().float() for k_ in gate_keys]) for ls in local_states]
        gate_spread = float(np.mean([(flat[i] - flat[j]).norm().item()
                                     for i in range(len(flat)) for j in range(i + 1, len(flat))]))
    res = dict(model=name, seed=seed, tag=tag, mode=MODE, cent_mode=CENT_MODE, dataset=DATASET_NAME, fed=FED,
               n_params=n_params, train_time=t_train, history=history, metrics=m, per_client=per_client,
               local_keys=local_keys, local_gate_mean_pairwise_distance=gate_spread, classes=classes,
               finished=datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
    _save(base, res, dict(y_true=yt, y_pred=yp, client=cid, gate_w=w if w is not None else np.zeros((0, 2))))
    log(f"<<< FED {name} | acc {m['accuracy']:.4f} | bal {m['balanced_accuracy']:.4f} | F1w {m['f1_weighted']:.4f} | "
        f"F1m {m['f1_macro']:.4f} | train {t_train:.0f}s")
    return res


def _fed_predict(model, name, clients, graphs, global_state, local_states):
    ys, ps, ws, cid = [], [], [], []
    for k, c in enumerate(clients):
        st = copy.deepcopy(global_state); st.update(local_states[k])
        model.load_state_dict(st)
        p, w = predict(model, graphs[k], c["test"])
        ys.append(c["test"]["y"].numpy()); ps.append(p); cid.append(np.full(len(p), k))
        if w is not None:
            ws.append(w)
    return np.concatenate(ys), np.concatenate(ps), (np.concatenate(ws) if ws else None), np.concatenate(cid)

## 7. Analysis functions

Tables, McNemar and paired t-tests, centrality effect, time to accuracy, per-class F1-scores and gate weights.

In [ ]:
def load_runs(tag):
    out = {}
    for fn in sorted(os.listdir(RUN_DIR)):
        if fn.startswith(tag + "__") and fn.endswith(".json"):
            with open(os.path.join(RUN_DIR, fn)) as f:
                r = json.load(f)
            out[(r["model"], r["seed"])] = r
    return out


def load_preds(tag, name, seed):
    return np.load(os.path.join(RUN_DIR, f"{tag}__{_safe(name)}__seed{seed}.npz"))


def table(runs, names):
    rows = []
    for name in names:
        rs = [runs[(name, s)] for s in SEEDS if (name, s) in runs]
        if not rs:
            continue
        def ms(f, scale=1.0, fmt="{:.2f}"):
            v = np.array([f(r) for r in rs]) * scale
            return (fmt + " ± " + fmt).format(v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0)
        rows.append(OrderedDict(model=name, n=len(rs),
                                acc=ms(lambda r: r["metrics"]["accuracy"], 100),
                                bal_acc=ms(lambda r: r["metrics"]["balanced_accuracy"], 100),
                                f1_w=ms(lambda r: r["metrics"]["f1_weighted"], 1, "{:.4f}"),
                                f1_m=ms(lambda r: r["metrics"]["f1_macro"], 1, "{:.4f}"),
                                prec_w=ms(lambda r: r["metrics"]["precision_weighted"], 1, "{:.4f}"),
                                fpr=ms(lambda r: r["metrics"]["fpr_macro"], 1, "{:.4f}"),
                                fnr=ms(lambda r: r["metrics"]["fnr_macro"], 1, "{:.4f}"),
                                train_s=ms(lambda r: r["train_time"], 1, "{:.0f}"),
                                params=rs[0]["n_params"]))
    return pd.DataFrame(rows).set_index("model") if rows else pd.DataFrame()


def mcnemar_p(y, a, b):
    ca, cb = a == y, b == y
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    if n01 + n10 == 0:
        return 1.0
    if n01 + n10 < 25:
        return float(binomtest(n01, n01 + n10, 0.5).pvalue)
    return float(chi2.sf((abs(n01 - n10) - 1) ** 2 / (n01 + n10), 1))


def significance(runs, tag, ours, baselines):
    rows = []
    for o in ours:
        for b in baselines:
            seeds = [s for s in SEEDS if (o, s) in runs and (b, s) in runs]
            if not seeds:
                continue
            ps, ga, gm, wins = [], [], [], 0
            for s in seeds:
                po, pb = load_preds(tag, o, s), load_preds(tag, b, s)
                assert (po["y_true"] == pb["y_true"]).all(), "different test sets"
                ps.append(mcnemar_p(po["y_true"], po["y_pred"], pb["y_pred"]))
                ga.append(100 * (runs[(o, s)]["metrics"]["accuracy"] - runs[(b, s)]["metrics"]["accuracy"]))
                gm.append(100 * (runs[(o, s)]["metrics"]["f1_macro"] - runs[(b, s)]["metrics"]["f1_macro"]))
                wins += int(ga[-1] > 0)
            tp = np.nan
            if len(seeds) > 1:
                tp = float(ttest_rel([runs[(o, s)]["metrics"]["accuracy"] for s in seeds],
                                     [runs[(b, s)]["metrics"]["accuracy"] for s in seeds]).pvalue)
            rows.append(dict(ours=o, baseline=b, acc_gain_pts=np.mean(ga), macroF1_gain_pts=np.mean(gm),
                             seeds_ours_better=f"{wins}/{len(seeds)}", mcnemar_max_p=max(ps),
                             mcnemar_all_p_lt_0_05=all(p < 0.05 for p in ps), paired_t_p=tp))
    return pd.DataFrame(rows)


def first_reach(hist, target, key):
    for h in hist:
        if h[key] >= target:
            return h["epoch"] if "epoch" in h else h["round"], h["cumulative_time"]
    return None, None


def time_to_accuracy(runs, ours, baselines):
    """Time for GTFusion to reach the FINAL test score of each baseline vs the time the baseline needed."""
    rows = []
    for key in ("test_accuracy", "test_balanced_accuracy"):
        for o in ours:
            for b in baselines:
                for s in SEEDS:
                    if (o, s) not in runs or (b, s) not in runs:
                        continue
                    target = runs[(b, s)]["history"][-1][key]
                    eb, tb = first_reach(runs[(b, s)]["history"], target, key)
                    eo, to = first_reach(runs[(o, s)]["history"], target, key)
                    rows.append(dict(metric=key.replace("test_", ""), ours=o, baseline=b, seed=s, target=target,
                                     baseline_epoch=eb, baseline_time=tb, ours_epoch=eo, ours_time=to))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    agg = df.groupby(["metric", "ours", "baseline"]).agg(
        target=("target", "mean"), baseline_epochs=("baseline_epoch", "mean"), baseline_time_s=("baseline_time", "mean"),
        ours_epochs=("ours_epoch", "mean"), ours_time_s=("ours_time", "mean"),
        reached=("ours_epoch", lambda v: f"{int(v.notna().sum())}/{len(v)}"))
    agg["time_saving_pct"] = 100 * (agg["baseline_time_s"] - agg["ours_time_s"]) / agg["baseline_time_s"]
    return agg


def centrality_effect(runs):
    """Full GTFusion vs the same model without centrality: final scores and epochs to reach fractions of the
    no-centrality model's final score."""
    rows = []
    for k in ("GAT", "SAGE"):
        on, off = f"GTFusion-{k}", f"GTFusion-{k} -centrality"
        for s in SEEDS:
            if (on, s) not in runs or (off, s) not in runs:
                continue
            for key in ("test_accuracy", "test_balanced_accuracy"):
                final_off = runs[(off, s)]["history"][-1][key]
                for frac in (0.98, 0.99, 1.0):
                    e_off, t_off = first_reach(runs[(off, s)]["history"], frac * final_off, key)
                    e_on, t_on = first_reach(runs[(on, s)]["history"], frac * final_off, key)
                    rows.append(dict(model=f"GTFusion-{k}", metric=key.replace("test_", ""), target=f"{frac} x final(off)",
                                     seed=s, epochs_off=e_off, epochs_on=e_on, time_off=t_off, time_on=t_on,
                                     final_off=100 * final_off, final_on=100 * runs[(on, s)]["history"][-1][key],
                                     centrality_time=runs[(on, s)]["centrality_time"],
                                     overhead_pct=100 * runs[(on, s)]["centrality_time"] / runs[(on, s)]["train_time"]))
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.groupby(["model", "metric", "target"]).agg(
        final_off=("final_off", "mean"), final_on=("final_on", "mean"), epochs_off=("epochs_off", "mean"),
        epochs_on=("epochs_on", "mean"), time_off_s=("time_off", "mean"), time_on_s=("time_on", "mean"),
        centrality_s=("centrality_time", "mean"), overhead_pct=("overhead_pct", "mean"))


def gate_per_class(runs):
    rows = []
    for k in ("GAT", "SAGE"):
        on, off = f"GTFusion-{k}", f"GTFusion-{k} -gate"
        seeds = [s for s in SEEDS if (on, s) in runs and (off, s) in runs]
        if not seeds:
            continue
        for c in runs[(on, seeds[0])]["classes"]:
            gains = [100 * (runs[(on, s)]["metrics"]["per_class"][c]["f1"] - runs[(off, s)]["metrics"]["per_class"][c]["f1"])
                     for s in seeds]
            wg = [runs[(on, s)]["gate"][c]["w_graph"] for s in seeds if runs[(on, s)]["gate"] and c in runs[(on, s)]["gate"]]
            rows.append(dict(model=on, cls=c, f1_with_gate=np.mean([runs[(on, s)]["metrics"]["per_class"][c]["f1"] for s in seeds]),
                             f1_without=np.mean([runs[(off, s)]["metrics"]["per_class"][c]["f1"] for s in seeds]),
                             gain_pts=np.mean(gains), seeds_gate_better=f"{sum(g > 0 for g in gains)}/{len(seeds)}",
                             w_graph=np.mean(wg) if wg else np.nan))
    return pd.DataFrame(rows).set_index(["model", "cls"]) if rows else pd.DataFrame()


def per_class_f1(runs, names):
    rows = {}
    for name in names:
        seeds = [s for s in SEEDS if (name, s) in runs]
        if not seeds:
            continue
        cls = runs[(name, seeds[0])]["classes"]
        rows[name] = {c: np.mean([runs[(name, s)]["metrics"]["per_class"][c]["f1"] for s in seeds]) for c in cls}
    return pd.DataFrame(rows).T

## 8. Sanity checks

Quick checks printed before training.

In [ ]:
def sanity_checks():
    banner("SANITY CHECKS")
    D = prepare_centralized(SEEDS[0])
    gp, gs, gc = D["graphs"]["plain"], D["graphs"]["subnet"], D["graphs"]["subnet_cent"]
    print(f"node features: plain {tuple(gp.x.shape)}, subnet {tuple(gs.x.shape)}, subnet+centrality {tuple(gc.x.shape)}")
    print(f"edges (directed): plain {gp.edge_index.shape[1]:,}, subnet {gs.edge_index.shape[1]:,}")
    assert gs.edge_index.shape[1] >= gp.edge_index.shape[1]
    assert torch.equal(gc.x[:, :8], gs.x[:, :8]) and torch.equal(gs.x, gp.x)
    assert gc.edge_attr.shape == (gc.edge_index.shape[1], 3)
    assert gp.edge_index.max().item() < D["n_nodes"] and gs.edge_index.max().item() < D["n_nodes"]
    assert torch.all(gc.x[-1] == 0), "padding node must be zero"
    cent = gc.x[:-1, 8:].numpy()
    print("centrality columns (degree, betweenness, PageRank) after scaling:")
    print(f"   std  {np.round(cent.std(0), 3)}   distinct values {[len(np.unique(np.round(cent[:, j], 6))) for j in range(3)]}")
    print(f"flow features: train {tuple(D['train']['x'].shape)}, mean {D['train']['x'].mean().item():+.4f}, "
          f"std {D['train']['x'].std().item():.4f}")
    print(f"class weights: {np.round(D['class_weights'].numpy(), 3).tolist()}")
    print(f"N-STGAT node sequences {tuple(gp.node_seq.shape)} | FedGATSage communities {gp.n_comm - 1}")
    node_dims = {k: g.x.shape[1] for k, g in D["graphs"].items()}
    for name in MODELS:
        m = build_model(name, node_dims, D["flow_dim"], D["n_classes"]).to(DEVICE).eval()
        g = model_graph(name, D["graphs"])
        with torch.no_grad():
            out = m(g, D["test"]["src"][:64].to(DEVICE), D["test"]["dst"][:64].to(DEVICE), D["test"]["x"][:64].to(DEVICE))
        assert out.shape == (64, D["n_classes"]) and torch.isfinite(out).all(), name
        print(f"   ok  {name:<36} graph={str(graph_key_of(name)):<12} params={sum(p.numel() for p in m.parameters()):,}")
    # baselines that must not see flow features: changing x must not change their output
    for name in ["GAT", "GraphSAGE", "FN-GNN", "Seq-GAT-LSTM"]:
        m = build_model(name, node_dims, D["flow_dim"], D["n_classes"]).to(DEVICE).eval()
        g = model_graph(name, D["graphs"])
        s, d, x = D["test"]["src"][:64].to(DEVICE), D["test"]["dst"][:64].to(DEVICE), D["test"]["x"][:64].to(DEVICE)
        with torch.no_grad():
            assert torch.allclose(m(g, s, d, x), m(g, s, d, torch.randn_like(x))), f"{name} uses flow features"
    print("   ok  GAT, GraphSAGE, FN-GNN, Seq-GAT-LSTM ignore flow features (graph only)")
    print("SANITY CHECKS PASSED")

## 9. Run all experiments

Runs every centralized and federated experiment. Finished runs are saved in `results/` and skipped when the cell is run again.

In [ ]:
T_START = time.time()
log(f"MODE {MODE} | centrality version {CENT_MODE} | seeds {SEEDS} | epochs {EPOCHS} | "
    f"federated {FED['rounds']} rounds x {FED['local_epochs']} local epochs | device {DEVICE} | "
    f"torch {torch.__version__} | output {os.path.abspath(OUT_DIR)}")
sanity_checks()

banner("CENTRALIZED RUNS")
ONE_SEED_ONLY = {"N-STGAT"} if (DATASET == "CIC-IDS-2017" and MODE == "FULL") else set()   # 14 h per 3 seeds otherwise
for seed in SEEDS:
    for name in MODELS:
        if name in ONE_SEED_ONLY and seed != SEEDS[0]:
            continue
        if name in _DS.get("skip", set()):
            continue
        run_central(name, seed)

banner("FEDERATED RUNS")
for seed in SEEDS:
    for name in FED_MODELS:
        if name in _DS.get("skip", set()):
            continue
        run_fed(name, seed)

## 10. Report: tables of the paper

Prints and saves every table of the paper as CSV files.

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 400)
CR, FR = load_runs("central"), load_runs("federated")
REPORT = {}

banner(f"1. MAIN TABLE, centralized ({MODE}, final epoch, mean ± std over {len(SEEDS)} seed(s))")
REPORT["main"] = table(CR, MAIN); print(REPORT["main"].to_string())

banner("2. SIGNIFICANCE: McNemar per seed on identical test flows, paired t-test over seeds")
REPORT["significance"] = significance(CR, "central", OURS, BASELINES)
print(REPORT["significance"].round(4).to_string(index=False))

banner("3. LEAVE-ONE-OUT ABLATION (what each part of our pipeline adds)")
for k in ("GAT", "SAGE"):
    REPORT[f"ablation_{k}"] = table(CR, ABLATION[k]); print(REPORT[f"ablation_{k}"].to_string()); print()

banner("4. CENTRALITY EFFECT: full GTFusion vs without centrality")
REPORT["centrality"] = centrality_effect(CR); print(REPORT["centrality"].round(3).to_string())

banner("5. TIME TO ACCURACY: time for GTFusion to reach each baseline's final test score")
REPORT["time_to_acc"] = time_to_accuracy(CR, OURS, BASELINES); print(REPORT["time_to_acc"].round(3).to_string())

banner("6. GATE PER CLASS: F1 with gate vs without (concatenation), and learned graph weight")
REPORT["gate_class"] = gate_per_class(CR); print(REPORT["gate_class"].round(4).to_string())

banner("7. PER-CLASS F1, main models")
REPORT["per_class"] = per_class_f1(CR, MAIN); print(REPORT["per_class"].round(4).to_string())

banner(f"8. FEDERATED ({FED['clients']} non-IID clients, alpha {FED['alpha']}, final round)")
REPORT["federated"] = table(FR, FED_MODELS); print(REPORT["federated"].to_string())
REPORT["fed_significance"] = significance(FR, "federated", OURS, [b for b in FED_MODELS if b not in OURS])
print(); print(REPORT["fed_significance"].round(4).to_string(index=False))

banner("9. FEDERATED: accuracy per client and personalised gate (graph weight per class, first seed)")
for name in OURS:
    r = FR.get((name, SEEDS[0]))
    if not r:
        continue
    print(f"\n{name}  (mean pairwise distance between client gates: {r['local_gate_mean_pairwise_distance']:.4f})")
    for ck, v in r["per_client"].items():
        gw = v["gate"] or {}
        pcs = ", ".join(f"{c} {x['w_graph']:.2f}" for c, x in gw.items() if c != "_overall")
        print(f"  {ck}: acc {v['accuracy']:.4f} | overall w_graph {gw.get('_overall', {}).get('w_graph', float('nan')):.3f} | {pcs}")
for name in FED_MODELS:
    r = FR.get((name, SEEDS[0]))
    if r and name not in OURS:
        print(f"  {name:<14} per-client acc: " + ", ".join(f"{v['accuracy']:.4f}" for v in r["per_client"].values()))

banner("11. SPEED: seconds per training epoch (same machine, same batch size) and final scores")
_rows = []
for name in MODELS:
    rs = [CR[(name, s)] for s in SEEDS if (name, s) in CR]
    if rs:
        _rows.append(dict(model=name, sec_per_epoch=np.mean([np.mean([h["epoch_time"] for h in r["history"]]) for r in rs]),
                          total_train_s=np.mean([r["train_time"] for r in rs]), params=rs[0]["n_params"],
                          acc=100 * np.mean([r["metrics"]["accuracy"] for r in rs]),
                          bal_acc=100 * np.mean([r["metrics"]["balanced_accuracy"] for r in rs])))
REPORT["speed"] = pd.DataFrame(_rows).set_index("model")
for ref in ("GAT", "GraphSAGE"):
    if ref in REPORT["speed"].index:
        REPORT["speed"][f"faster_than_{ref}_pct"] = 100 * (1 - REPORT["speed"]["sec_per_epoch"] / REPORT["speed"].loc[ref, "sec_per_epoch"])
print(REPORT["speed"].round(2).to_string())

banner("12. MATCHED DEPTH: GTFusion vs a standalone GNN with the SAME depth and heads")
_rows = []
for ours, base in (("GTFusion-GAT", "GAT-3L"), ("GTFusion-SAGE", "GraphSAGE-3L")):
    for s in SEEDS:
        if (ours, s) not in CR or (base, s) not in CR:
            continue
        ro, rb = CR[(ours, s)], CR[(base, s)]
        for key in ("test_accuracy", "test_balanced_accuracy"):
            tgt = rb["history"][-1][key]
            eb, tb = first_reach(rb["history"], tgt, key)
            eo, to = first_reach(ro["history"], tgt, key)
            _rows.append(dict(ours=ours, baseline=base, metric=key.replace("test_", ""), seed=s,
                              sec_epoch_base=np.mean([h["epoch_time"] for h in rb["history"]]),
                              sec_epoch_ours=np.mean([h["epoch_time"] for h in ro["history"]]),
                              final_base=100 * tgt, final_ours=100 * ro["history"][-1][key],
                              epochs_base=eb, epochs_ours=eo, time_base=tb, time_ours=to))
_md = pd.DataFrame(_rows)
if not _md.empty:
    _md = _md.groupby(["ours", "baseline", "metric"]).mean(numeric_only=True).drop(columns="seed")
    _md["per_epoch_cost_pct"] = 100 * (_md["sec_epoch_ours"] / _md["sec_epoch_base"] - 1)
    _md["time_to_baseline_final_saving_pct"] = 100 * (1 - _md["time_ours"] / _md["time_base"])
    REPORT["matched_depth"] = _md
    print(_md.round(3).to_string())
    print()
    print(table(CR, ["GAT-3L", "GTFusion-GAT", "GraphSAGE-3L", "GTFusion-SAGE"]).to_string())

if MODE == "FAST":
    banner("10. PREDICTED FULL RUN TIME (epoch times measured here, scaled to the full data, 20 epochs, 3 seeds)")
    scale, rows = 1.0 / SAMPLE_FRAC, []
    full_models = [m for m in MODELS if not m.endswith(f"cent={ALT_CENT}")]
    for name in full_models:
        r = CR.get((name, SEEDS[0]))
        if r:
            ep = np.mean([h["epoch_time"] for h in r["history"]])
            rows.append(dict(part="centralized", model=name, measured_s_per_epoch=ep,
                             predicted_hours=ep * scale * 20 * 3 / 3600))
    for name in FED_MODELS:
        r = FR.get((name, SEEDS[0]))
        if r:
            per_round = r["train_time"] / len(r["history"])
            rows.append(dict(part="federated", model=name, measured_s_per_epoch=per_round,
                             predicted_hours=per_round * scale * 30 * 3 / 3600))
    REPORT["predicted_time"] = pd.DataFrame(rows)
    print(REPORT["predicted_time"].round(3).to_string(index=False))
    print(f"\nPREDICTED TOTAL FOR FULL: {REPORT['predicted_time']['predicted_hours'].sum():.1f} hours "
          f"(plus evaluation, roughly +10%)")

for k, v in REPORT.items():
    if isinstance(v, pd.DataFrame) and not v.empty:
        v.to_csv(os.path.join(OUT_DIR, f"report_{k}.csv"))
banner(f"ALL DONE ({MODE}) in {(time.time() - T_START) / 60:.1f} min. Tables saved in {os.path.abspath(OUT_DIR)}")